# 1. Import Data

In [1]:
# Standard libs
from __future__ import annotations
from dataclasses import dataclass
from typing import Optional, Tuple, List
import os
import glob
import json
import argparse
from pathlib import Path
from typing import Optional
from typing import Optional, Dict, Any
from itertools import product
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import List, Optional, Dict, Any, Tuple
import torch.multiprocessing as mp


# Utils
import math
import random
import gc
import sys

# Data & Processing
import numpy as np
import pandas as pd
import geopandas as gpd

# Scikit-Learn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, confusion_matrix
from sklearn.neighbors import BallTree, NearestNeighbors
from sklearn.model_selection import train_test_split

# TQDM
from tqdm.notebook import tqdm
from joblib import Parallel, delayed

# Plotting
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader

# PyTorch Geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GatedGraphConv, GATConv, ChebConv


# Confirm the current working directory
print("Current working directory:", os.getcwd())

Current working directory: /home/qusta100/STGNN/02_Notebooks/03_Window_Building


In [2]:
# Load main data set
df = pd.read_csv(
    "/gpfs/scratch/qusta100/STGNN/01_Data/02_Processed_Data/02_Analysis/final_df.csv",
    dtype={9: str}
)
df["date"] = pd.to_datetime(df["date"])

# 2. Prepare Data

In [3]:
# =============================
# a) Parameters
# =============================

STATION_COL = "station_uuid"
LAT_COL = "latitude"
LON_COL = "longitude"
TIME_COL = "date"

# Default-Targets
TARGET_COLS_DEFAULT = ["e5"]
FEATURE_COLS_DEFAULT = ["e5"]
BRAND_COL = "brand"

RADIUS_KM = 10
NEIGHBOUR = 5

SEED = 42
EMBED_DIM = 32
HIDDEN_DIM = 64
ATTN_HEADS_DEFAULT = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS = 100
PATIENCE = 10
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

WINDOW_SIZE = 672 # high value due to different encoder
HORIZON_STEPS = 4  # Horizont per Target
LENGTH_ENCODER_DEFAULT = 16 # Target Length after Strides


# =============================
# b) Helper functions
# =============================

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def km_to_radians(km: float) -> float:
    earth_radius_km = 6371.0088
    return km / earth_radius_km


def build_knn_graph(
    lat_lon_deg: np.ndarray,
    k: int = 5,
    return_dists: bool = False,
    RADIUS_KM: Optional[float] = None
) -> np.ndarray | Tuple[np.ndarray, np.ndarray]:
    
    lat_lon_rad = np.radians(lat_lon_deg)
    tree = BallTree(lat_lon_rad, metric="haversine")

    dist_rad, ind = tree.query(lat_lon_rad, k=k + 1)

    src, dst, dists_km = [], [], []
    radius_rad = km_to_radians(RADIUS_KM) if RADIUS_KM is not None else None

    for i, (neigh_idx, neigh_dist_rad) in enumerate(zip(ind, dist_rad)):
        for j, dij_rad in zip(neigh_idx[1:], neigh_dist_rad[1:]):
            if radius_rad is not None and dij_rad > radius_rad:
                continue

            dij_km = dij_rad * 6371.0088  # Conversion for output
            src.append(j)
            dst.append(i)
            dists_km.append(dij_km)

    edge_index = np.vstack([np.array(src, dtype=int), np.array(dst, dtype=int)])

    if return_dists:
        return edge_index, np.asarray(dists_km, dtype=float)
    return edge_index


# =============================
# c) Window-based ST data (multi-step, multi-target)
# =============================

@dataclass
class STWindowDataMulti:
    x: torch.Tensor             # [T, N, F] full time series, F = num_features
    x_mask: torch.Tensor   
    y: torch.Tensor             # [T, N, H * num_targets] H Horizonte für alle Targets
    edge_index: torch.Tensor
    edge_weight: torch.Tensor 
    train_end_idx: np.ndarray   # time indices where windows end (train)
    val_end_idx: np.ndarray     # time indices (validation)
    test_end_idx: np.ndarray    # time indices (test)
    valid_nodes: np.ndarray     # node indices (z.B. Thüringen)
    times: np.ndarray           # sorted timestamps
    meta: pd.DataFrame          # station metadata (uuid, lat, lon, node_id)
    feature_cols: List[str]     # relevant Feature-Spalten
    target_cols: List[str]      # relevant Target-Spalten
    time_feat: np.ndarray       # [T, time_in_dim], z.B. weekday, holiday, time_sin
    condat: Optional[torch.Tensor]  # [N, cond_in_dim]


def make_window_data_multi_from_df(
    df: pd.DataFrame,
    window_size: int = WINDOW_SIZE,
    horizon_steps: int = HORIZON_STEPS,
    feature_cols: Optional[List[str]] = None,
    target_cols: Optional[List[str]] = None,
    stride: int = 1, 
    neighbour_k: int = NEIGHBOUR,          # neu
    radius_km: Optional[float] = RADIUS_KM
) -> STWindowDataMulti:

    if target_cols is None:
        target_cols = TARGET_COLS_DEFAULT
    if feature_cols is None:
        feature_cols = FEATURE_COLS_DEFAULT

    # Necessary Columns
    base_cols = [STATION_COL, LAT_COL, LON_COL, TIME_COL, "in_thuringia", BRAND_COL]
    all_val_cols = sorted(set(feature_cols) | set(target_cols))
    needed = base_cols + all_val_cols
    assert all(c in df.columns for c in needed), f"Missing columns: {set(needed) - set(df.columns)}"

    df = df.copy()

    # Stations + node ids
    stations = (
        df[[STATION_COL, LAT_COL, LON_COL, BRAND_COL]]
        .drop_duplicates(subset=[STATION_COL])
        .reset_index(drop=True)
    )
    stations["node_id"] = np.arange(len(stations))
    station2id = stations.set_index(STATION_COL)["node_id"].to_dict()

    # Time axis
    times = np.sort(df[TIME_COL].unique())
    time2id = {t: i for i, t in enumerate(times)}
    
    # ---- Build time features: weekday, holiday, time_sin ----
    
    weekday_cols = [f"wd_{i}" for i in range(7)]
    time_cols = [f"t_{i}" for i in range(96)]
    
    time_df = (
        df[[TIME_COL, *weekday_cols, "holiday", *time_cols, "Brent_Price"]]
        .drop_duplicates(subset=[TIME_COL])
        .sort_values(TIME_COL)
    )

    assert np.array_equal(time_df[TIME_COL].to_numpy(), times), \
        "The timelines of time_df and times do not match."

    time_feat = time_df[[*weekday_cols, "holiday", *time_cols, "Brent_Price"]].to_numpy(dtype=float)

    N = len(stations)
    T = len(times)
    M = len(target_cols)             # Number Targets
    F = len(feature_cols)            # Number Features
    H = horizon_steps
    
    
    # ---- condat aus brand bauen (One-Hot) ----
    brands = stations[BRAND_COL].astype("category")
    brand_codes = brands.cat.codes.to_numpy()          # [N]
    n_brands = len(brands.cat.categories)

    brand_oh = np.eye(n_brands, dtype=np.float32)[brand_codes]  # [N, n_brands]
    condat = torch.tensor(brand_oh, dtype=torch.float32)         # [N, cond_in_dim]
    

    # Spatial graph
    lat_lon = stations[[LAT_COL, LON_COL]].to_numpy(dtype=float)
    edge_index_np, edge_dist_km = build_knn_graph(
        lat_lon_deg=lat_lon,
        k=neighbour_k,
        return_dists=True,
        RADIUS_KM=radius_km
    )
    edge_index = torch.tensor(edge_index_np, dtype=torch.long)
    
    # Distance -> Weight: the closer, the greater
    sigma = np.median(edge_dist_km)
    edge_weight_np = np.exp(-(edge_dist_km**2) / (2*sigma**2))
    edge_weight = torch.tensor(edge_weight_np, dtype=torch.float32)

    # Tensors
    x = torch.zeros((T, N, F), dtype=torch.float32)                    # [T, N, F]
    x_mask = torch.zeros((T, N, F), dtype=torch.float32) 
    y = torch.full((T, N, H * M), float("nan"), dtype=torch.float32)   # [T, N, H*M]

    # Aggregation per (station, time) across all relevant columns
    df_idx = (
        df[[STATION_COL, TIME_COL] + all_val_cols]
        .groupby([STATION_COL, TIME_COL], as_index=False)
        .mean()
        .sort_values(TIME_COL)
    )

    df_idx["node_id"] = df_idx[STATION_COL].map(station2id)
    df_idx["time_id"] = df_idx[TIME_COL].map(time2id)

    # x[t, n, f] = Features at time t
    for _, row in df_idx.iterrows():
        t = int(row["time_id"])
        n = int(row["node_id"])
        for f_idx, col in enumerate(feature_cols):
            v = row[col]
            if pd.notna(v):
                x[t, n, f_idx] = float(v)
                x_mask[t, n, f_idx] = 1.0

    # y[t, n, m*H + h] = Target m at time t+h+1
    for station, g in df_idx.groupby(STATION_COL):
        g = g.sort_values("time_id")
        node_id = int(g["node_id"].iloc[0])
        t_ids = g["time_id"].to_numpy()

        L = len(t_ids)
        for m, col in enumerate(target_cols):
            vals = g[col].to_numpy()

            for idx in range(L):
                t = int(t_ids[idx])
                if idx + H >= L:
                    break

                future_vals = vals[idx + 1: idx + 1 + H]

                if np.any(np.isnan(future_vals)):
                    continue

                for h in range(H):
                    y[t, node_id, m * H + h] = float(future_vals[h])

    # Time indices where full horizon is available
    effective_T = T - H
    if effective_T < window_size:
        raise ValueError(f"Too few timesteps ({effective_T}) for window size {window_size} and horizon {H}.")

    # Time-based split over 0..effective_T-1
    time_indices = np.arange(effective_T)
    num_total = len(time_indices)
    num_train = int((1.0 - VAL_SPLIT - TEST_SPLIT) * num_total)
    num_val   = int(VAL_SPLIT * num_total)
    num_test  = num_total - num_train - num_val

    train_t = time_indices[:num_train]
    val_t   = time_indices[num_train:num_train + num_val]
    test_t  = time_indices[num_train + num_val:]

    # Window end indices
    min_end = window_size - 1
    train_end_idx = train_t[train_t >= min_end]
    val_end_idx   = val_t[val_t >= min_end]
    test_end_idx  = test_t[test_t >= min_end]
    
    # stride
    train_end_idx = train_end_idx[::stride]
    val_end_idx   = val_end_idx[::stride]
    test_end_idx  = test_end_idx[::stride]

    # Thuringia mask
    th_mask = (
        df[[STATION_COL, "in_thuringia"]]
        .drop_duplicates()
        .set_index(STATION_COL)["in_thuringia"]
    )
    stations = stations.join(th_mask, on=STATION_COL)
    valid_nodes = stations.index[stations["in_thuringia"] == 1].to_numpy()

    if valid_nodes.size == 0:
        raise ValueError("No nodes with in_thuringia == 1 found.")

    meta = stations[[STATION_COL, LAT_COL, LON_COL, "node_id"]].copy()

    return STWindowDataMulti(
        x=x,
        x_mask=x_mask,
        y=y,
        edge_index=edge_index,
        edge_weight=edge_weight, 
        train_end_idx=train_end_idx,
        val_end_idx=val_end_idx,
        test_end_idx=test_end_idx,
        valid_nodes=valid_nodes,
        times=times,
        meta=meta,
        feature_cols=feature_cols,
        target_cols=target_cols,
        time_feat=time_feat,
        condat=condat,
    )

# 3. Create Data Windows

## 3.1. Data Windows E5

In [4]:
# 4 Neighbours
data_E5_4N = make_window_data_multi_from_df(df, neighbour_k=4, feature_cols=["e5"], target_cols=["e5"])
torch.save({
    "x": data_E5_4N.x,
    "x_mask": data_E5_4N.x_mask,
    "y": data_E5_4N.y,
    "edge_index": data_E5_4N.edge_index,
    "edge_weight": data_E5_4N.edge_weight,
    "time_feat": data_E5_4N.time_feat,
    "train_end_idx": data_E5_4N.train_end_idx,
    "val_end_idx": data_E5_4N.val_end_idx,
    "test_end_idx": data_E5_4N.test_end_idx,
    "valid_nodes": data_E5_4N.valid_nodes,
    "target_cols": data_E5_4N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/01_E5/01_Dist/data_Dist_4N.pt")

In [5]:
# 5 Neighbours
data_E5_5N = make_window_data_multi_from_df(df, neighbour_k=5, feature_cols=["e5"], target_cols=["e5"])
torch.save({
    "x": data_E5_5N.x,
    "x_mask": data_E5_5N.x_mask,
    "y": data_E5_5N.y,
    "edge_index": data_E5_5N.edge_index,
    "edge_weight": data_E5_5N.edge_weight,
    "time_feat": data_E5_5N.time_feat,
    "train_end_idx": data_E5_5N.train_end_idx,
    "val_end_idx": data_E5_5N.val_end_idx,
    "test_end_idx": data_E5_5N.test_end_idx,
    "valid_nodes": data_E5_5N.valid_nodes,
    "target_cols": data_E5_5N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/01_E5/01_Dist/data_Dist_5N.pt")

In [6]:
# 6 Neighbours
data_E5_6N = make_window_data_multi_from_df(df, neighbour_k=6, feature_cols=["e5"], target_cols=["e5"])
torch.save({
    "x": data_E5_6N.x,
    "x_mask": data_E5_6N.x_mask,
    "y": data_E5_6N.y,
    "edge_index": data_E5_6N.edge_index,
    "edge_weight": data_E5_6N.edge_weight,
    "time_feat": data_E5_6N.time_feat,
    "train_end_idx": data_E5_6N.train_end_idx,
    "val_end_idx": data_E5_6N.val_end_idx,
    "test_end_idx": data_E5_6N.test_end_idx,
    "valid_nodes": data_E5_6N.valid_nodes,
    "target_cols": data_E5_6N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/01_E5/01_Dist/data_Dist_6N.pt")

## 3.2. Data Windows E10

In [7]:
# 4 Neighbours
data_E10_4N = make_window_data_multi_from_df(df, neighbour_k=4, feature_cols=["e10"], target_cols=["e10"])
torch.save({
    "x": data_E10_4N.x,
    "x_mask": data_E10_4N.x_mask,
    "y": data_E10_4N.y,
    "edge_index": data_E10_4N.edge_index,
    "edge_weight": data_E10_4N.edge_weight,
    "time_feat": data_E10_4N.time_feat,
    "train_end_idx": data_E10_4N.train_end_idx,
    "val_end_idx": data_E10_4N.val_end_idx,
    "test_end_idx": data_E10_4N.test_end_idx,
    "valid_nodes": data_E10_4N.valid_nodes,
    "target_cols": data_E10_4N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/02_E10/01_Dist/data_Dist_4N.pt")

In [8]:
# 5 Neighbours
data_E10_5N = make_window_data_multi_from_df(df, neighbour_k=5, feature_cols=["e10"], target_cols=["e10"])
torch.save({
    "x": data_E10_5N.x,
    "x_mask": data_E10_5N.x_mask,
    "y": data_E10_5N.y,
    "edge_index": data_E10_5N.edge_index,
    "edge_weight": data_E10_5N.edge_weight,
    "time_feat": data_E10_5N.time_feat,
    "train_end_idx": data_E10_5N.train_end_idx,
    "val_end_idx": data_E10_5N.val_end_idx,
    "test_end_idx": data_E10_5N.test_end_idx,
    "valid_nodes": data_E10_5N.valid_nodes,
    "target_cols": data_E10_5N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/02_E10/01_Dist/data_Dist_5N.pt")

In [9]:
# 6 Neighbours
data_E10_6N = make_window_data_multi_from_df(df, neighbour_k=6, feature_cols=["e10"], target_cols=["e10"])
torch.save({
    "x": data_E10_6N.x,
    "x_mask": data_E10_6N.x_mask,
    "y": data_E10_6N.y,
    "edge_index": data_E10_6N.edge_index,
    "edge_weight": data_E10_6N.edge_weight,
    "time_feat": data_E10_6N.time_feat,
    "train_end_idx": data_E10_6N.train_end_idx,
    "val_end_idx": data_E10_6N.val_end_idx,
    "test_end_idx": data_E10_6N.test_end_idx,
    "valid_nodes": data_E10_6N.valid_nodes,
    "target_cols": data_E10_6N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/02_E10/01_Dist/data_Dist_6N.pt")

## 3.3. Data Windows Diesel

In [10]:
# 4 Neighbours
data_Diesel_4N = make_window_data_multi_from_df(df, neighbour_k=4, feature_cols=["diesel"], target_cols=["diesel"])
torch.save({
    "x": data_Diesel_4N.x,
    "x_mask": data_Diesel_4N.x_mask,
    "y": data_Diesel_4N.y,
    "edge_index": data_Diesel_4N.edge_index,
    "edge_weight": data_Diesel_4N.edge_weight,
    "time_feat": data_Diesel_4N.time_feat,
    "train_end_idx": data_Diesel_4N.train_end_idx,
    "val_end_idx": data_Diesel_4N.val_end_idx,
    "test_end_idx": data_Diesel_4N.test_end_idx,
    "valid_nodes": data_Diesel_4N.valid_nodes,
    "target_cols": data_Diesel_4N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/03_Diesel/01_Dist/data_Dist_4N.pt")

In [11]:
# 5 Neighbours
data_Diesel_5N = make_window_data_multi_from_df(df, neighbour_k=5, feature_cols=["diesel"], target_cols=["diesel"])
torch.save({
    "x": data_Diesel_5N.x,
    "x_mask": data_Diesel_5N.x_mask,
    "y": data_Diesel_5N.y,
    "edge_index": data_Diesel_5N.edge_index,
    "edge_weight": data_Diesel_5N.edge_weight,
    "time_feat": data_Diesel_5N.time_feat,
    "train_end_idx": data_Diesel_5N.train_end_idx,
    "val_end_idx": data_Diesel_5N.val_end_idx,
    "test_end_idx": data_Diesel_5N.test_end_idx,
    "valid_nodes": data_Diesel_5N.valid_nodes,
    "target_cols": data_Diesel_5N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/03_Diesel/01_Dist/data_Dist_5N.pt")

In [12]:
# 6 Neighbours
data_Diesel_6N = make_window_data_multi_from_df(df, neighbour_k=6, feature_cols=["diesel"], target_cols=["diesel"])
torch.save({
    "x": data_Diesel_6N.x,
    "x_mask": data_Diesel_6N.x_mask,
    "y": data_Diesel_6N.y,
    "edge_index": data_Diesel_6N.edge_index,
    "edge_weight": data_Diesel_6N.edge_weight,
    "time_feat": data_Diesel_6N.time_feat,
    "train_end_idx": data_Diesel_6N.train_end_idx,
    "val_end_idx": data_Diesel_6N.val_end_idx,
    "test_end_idx": data_Diesel_6N.test_end_idx,
    "valid_nodes": data_Diesel_6N.valid_nodes,
    "target_cols": data_Diesel_6N.target_cols,
}, "/home/qusta100/STGNN/01_Data/02_Processed_Data/03_Data_Windows/03_Diesel/01_Dist/data_Dist_6N.pt")